# PDF ingestion pipeline debugger



In [ ]:
from pathlib import Path
import json
import os
import sys
import time
from types import SimpleNamespace

current_folder = Path.to_cwd() if hasattr(Path, 'to_cwd') else Path.cwd()
REPO_ROOT =current_folder.parent

print(REPO_ROOT)
if not (REPO_ROOT / 'backend').is_dir():
    raise RuntimeError('Start Jupyter from the repository root.')

BACKEND_ROOT = REPO_ROOT / 'backend'
sys.path.insert(0, str(BACKEND_ROOT))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.settings')
import django
django.setup()

from ingestion.pipeline.base import BasePipelineStep
from ingestion.pipeline.steps.pdf_extraction import PDFExtractionStep
from ingestion.pipeline.steps.line_reconstruction import LineReconstructionStep
from ingestion.pipeline.steps.page_classification import PageClassificationStep
from ingestion.pipeline.steps.metadata_front_matter import MetadataFrontMatterStep
from ingestion.pipeline.steps.metadata_people import MetadataPeopleStep
from ingestion.pipeline.steps.metadata_sections import MetadataSectionsStep
from ingestion.pipeline.utils.heading_utils import looks_like_heading
from ingestion.pipeline.utils.text_cleaners import TOC_ENTRY, canonicalize_heading, normalize_line


BasePipelineStep.log = lambda self, job, message, level='INFO': print(f'[{self.step_name}] {message}')


In [ ]:

PDF_PATH = REPO_ROOT / 'docs' / 'docs' /  '4D Gaussian Splatting.pdf'
# Example: PDF_PATH = Path('/absolute/path/to/report.pdf')

if not PDF_PATH.is_file():
    raise FileNotFoundError(f'Set PDF_PATH to a real PDF first: {PDF_PATH}')

OUTPUT_DIR = REPO_ROOT / 'notebook_outputs' / PDF_PATH.stem
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def write_json(filename, value):
    path = OUTPUT_DIR / filename
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    print(f'Wrote {path.relative_to(REPO_ROOT)}')

# The step API only needs job.file.path and file_hash here. It is not a saved Django model.
job = SimpleNamespace(file=SimpleNamespace(path=str(PDF_PATH)), file_hash='notebook-debug')
context = {'start_time': time.perf_counter(), 'draft_metadata': {}}



In [ ]:
# Layout extraction → reconstructed lines → page classification
for step, context_key, filename in [
    (PDFExtractionStep(), 'raw_pages', '01_raw_pages.json'),
    (LineReconstructionStep(), 'reconstructed_pages', '02_reconstructed_pages.json'),
    (PageClassificationStep(), 'classified_pages', '03_classified_pages.json'),
]:
    context = step.run(job, context)
    write_json(filename, context[context_key])

page_summary = [
    {
        'page': page['page_number'],
        'page_type': page.get('page_type'),
        'lines': len(page.get('lines', [])),
        'preview': ' '.join(line['text'] for line in page.get('lines', [])[:3])[:160],
    }
    for page in context['classified_pages']
]
write_json('04_page_classification_summary.json', page_summary)
page_summary


In [ ]:
# Existing front-matter and people extraction, still without any database writes.
for step in (MetadataFrontMatterStep(), MetadataPeopleStep()):
    context = step.run(job, context)

front_matter_debug = {
    'project_fields': context['draft_metadata'].get('project_fields', {}),
    'keywords': context['draft_metadata'].get('keywords', []),
    'people_and_roles': context['draft_metadata'].get('people_and_roles', []),
}
write_json('05_front_matter_and_people.json', front_matter_debug)
front_matter_debug


In [ ]:
# Existing back-matter extraction is useful independently of the section-tree experiment.
back_matter = MetadataSectionsStep.extract_back_matter(context['classified_pages'])
write_json('06_back_matter.json', back_matter)
{
    'references_pages': (back_matter['references']['page_start'], back_matter['references']['page_end']),
    'reference_characters': len(back_matter['references']['content']),
    'appendices': [item['title'] for item in back_matter['appendices']],
}


## Flat section experiment

This intentionally accepts only known major section names. It ignores all `2.2`-style subheadings, equations, and figure labels, and stops before References/Appendix. Added local names to `MAJOR_SECTION_ALIASES` after inspecting a document.

In [ ]:
import re

# Flat sections use a strict first pass, then a controlled recovery pass for
# reports whose genuine bold headings are smaller than 14 pt.
MIN_MAJOR_HEADING_FONT_SIZE = 14.0
MIN_RECOVERY_HEADING_FONT_SIZE = 11.0
TOC_CONTINUATION_DOT_LEADER_LINES = 1
TOC_DOT_LEADER = re.compile(r'\.{2,}\s*(?:[ivxlcdm]+|\d+|[a-z])?\s*$', re.I)


MAJOR_SECTION_ALIASES = {
    # Introduction and context
    
    'background': 'Background',
    'context': 'Background',
    'introduction': 'Introduction',
    'introduction and background': 'Introduction',
    'problem background': 'Background',
    'problem context': 'Background',
    
    'problem statement': 'Problem Statement',
    'research background': 'Background',

    # Problem, objectives, and scope
   
    'objectives': 'Objectives',
    'project objectives': 'Objectives',
    'research objectives': 'Objectives',
    'scope': 'Scope',
    'scope and limitations': 'Scope',
    'scope of the study': 'Scope',
    'significance of the study': 'Significance',
  
    # Literature and related work
    'literature review': 'Literature Review',
    'literature survey': 'Literature Review',
    'related literature': 'Literature Review',
    'related studies': 'Related Work',
    'related work': 'Related Work',
    'review of literature': 'Literature Review',
  
    # Analysis, requirements, and design
    'analysis': 'Analysis',
    'requirement analysis': 'Requirements Analysis',
    'requirements analysis': 'Requirements Analysis',
    'requirements specification': 'Requirements',
    'system analysis': 'System Analysis',
    'system architecture': 'System Design',
    'system design': 'System Design',
    'system overview': 'System Design',
    'system specification': 'System Design',

    # Methodology and approach
    'project approach': 'Methodology',
    'methodology': 'Methodology',
    'methodology and design': 'Methodology',
    'proposed approach': 'Methodology',
    'proposed methodology': 'Methodology',
    'research methodology': 'Methodology',
    'research design': 'Methodology',
    'technical approach': 'Methodology',
    'method': 'Methodology',

    # Proposed system and implementation
    'implementation': 'Implementation',
    'implementation details': 'Implementation',
    'implementation methodology': 'Implementation',
    'proposed system': 'Proposed System',
    'system implementation': 'Implementation',
    'system development': 'Implementation',

    # Experimental setup and evaluation
    'experimental setup': 'Experiments',
    'experimental methodology': 'Experiments',
    'experimental results': 'Results',
    'experiments': 'Experiments',
    'experiments and results': 'Results',
    'experiments and evaluation': 'Evaluation',
    'performance analysis': 'Evaluation',
    'performance evaluation': 'Evaluation',
    'proposed experimental setup': 'Experiments',
    'testing': 'Testing',
    'test results': 'Results',
    'validation': 'Validation',

    # Results and discussion
    'analysis and discussion': 'Discussion',
    'discussion': 'Discussion',
    'discussion and analysis': 'Discussion',
    'findings': 'Results',
    'results': 'Results',
    'results and analysis': 'Results',
    'results and discussion': 'Results',
    'results and findings': 'Results',

    # Evaluation and comparison
   
    'comparative analysis': 'Evaluation',
    'evaluation': 'Evaluation',
    'evaluation and analysis': 'Evaluation',
    'evaluation results': 'Evaluation',
    'performance comparison': 'Evaluation',
    'system evaluation': 'Evaluation',

    # Limitations and future work
    'future enhancement': 'Future Enhancement',
    'future enhancements': 'Future Enhancement',
    'future improvement': 'Future Enhancement',
    'future improvements': 'Future Enhancement',
    'future scope': 'Future Work',
    'future work': 'Future Work',
    'limitations': 'Limitations',
    'limitations and future enhancements': 'Future Enhancement',
    'limitations and future work': 'Future Work',

    # Conclusion and recommendations
    'conclusion': 'Conclusion',
    'conclusion and future work': 'Conclusion',
    'conclusions': 'Conclusion',
    'concluding remarks': 'Conclusion',
    'final remarks': 'Conclusion',
    'recommendation': 'Recommendations',
    'recommendations': 'Recommendations',
    'summary': 'Conclusion',
    'summary and conclusion': 'Conclusion',
    'summary and conclusions': 'Conclusion',
}


IGNORED_PAGE_TYPES = {
    'abstract', 'acknowledgement', 'approval', 'cover', 'copyright', 'toc',
    'declaration', 'certificate', 'list_of_figures', 'list_of_tables',
    'list_of_abbreviations',
}

def normalized_major_heading(text):
    text = normalize_line(text)
    text = re.sub(r'^chapter\s+(?:[ivxlcdm]+|\d+)\b[. :\-]*', '', text, flags=re.I)
    text = re.sub(r'^\d+(?:\.\d+)*[.) :\-]+', '', text)
    return text.strip().rstrip(':')

def line_font_size(line):
    try:
        return float(line.get('font_size', 0))
    except (TypeError, ValueError):
        return 0

def is_strict_major_heading(line, candidate, body_font_size):
    """Primary rule: a canonical heading must be bold and at least 14 pt."""
    return (
        bool(line.get('is_bold', False))
        and line_font_size(line) >= MIN_MAJOR_HEADING_FONT_SIZE
        and looks_like_heading(candidate)
    )

def is_recovery_major_heading(line, candidate, body_font_size):
    """Looser rule, used only after the strict pass finds <= 2 sections."""
    font_size = line_font_size(line)
    return (
        looks_like_heading(candidate)
        and font_size >= MIN_RECOVERY_HEADING_FONT_SIZE
        and (bool(line.get('is_bold', False)) or font_size >= body_font_size + 1.0)
    )

def content_body_font_size(pages):
    """Median non-bold text size on content pages, used only by recovery."""
    sizes = sorted(
        line_font_size(line)
        for page in pages
        if page.get('page_type') == 'content'
        for line in page.get('lines', [])
        if normalize_line(line.get('text', '')) and not line.get('is_bold', False)
        and line_font_size(line) > 0
    )
    return sizes[len(sizes) // 2] if sizes else 10.0

def toc_dot_leader_count(page):
    return sum(
        bool(TOC_DOT_LEADER.search(normalize_line(line.get('text', ''))))
        for line in page.get('lines', [])
    )

def is_toc_continuation_page(page):
    """A continuation page has at least one TOC-style dot-leader row."""
    return toc_dot_leader_count(page) >= TOC_CONTINUATION_DOT_LEADER_LINES

def is_confirmed_back_matter_heading(line, text, heading_rule, body_font_size):
    """Do not let an unclassified TOC row such as '39 References' stop extraction."""
    kind = MetadataSectionsStep._back_matter_heading(text)
    if not kind:
        return None
    candidate = normalized_major_heading(text)
    return kind if heading_rule(line, candidate, body_font_size) else None

def content_pages_with_toc_guard(pages):
    in_toc_sequence = False
    for page in pages:
        if page.get('page_type') == 'toc':
            in_toc_sequence = True
            continue
        if in_toc_sequence and is_toc_continuation_page(page):
            # No page limit: third/fourth TOC pages stay excluded while
            # their dot-leader rows continue.
            continue
        # Only content pages are eligible to create or contribute to sections.
        if page.get('page_type') == 'content':
            in_toc_sequence = False
            yield page

def extract_flat_sections(
    pages, heading_rule, stop_at_back_matter=True, include_unsectioned_content=False
):
    sections, current, in_back_matter = [], None, False
    body_font_size = content_body_font_size(pages)
    for page in content_pages_with_toc_guard(pages):
        page_number = page.get('page_number', page.get('page_num'))
        for line in page.get('lines', []):
            text = normalize_line(line.get('text', ''))
            if not text or TOC_ENTRY.search(text):
                continue
            back_matter_kind = is_confirmed_back_matter_heading(
                line, text, heading_rule, body_font_size
            )
            if stop_at_back_matter and back_matter_kind:
                in_back_matter = True
                current = None
                continue
            if in_back_matter:
                continue

            candidate = normalized_major_heading(text)
            canonical = MAJOR_SECTION_ALIASES.get(candidate.lower())
            if canonical and heading_rule(line, candidate, body_font_size):
                current = {
                    'section_name': canonical,
                    'source_heading': text,
                    'canonical_section': canonicalize_heading(candidate),
                    'page_start': page_number,
                    'page_end': page_number,
                    'content': '',
                }
                sections.append(current)
                continue
            if current:
                current['content'] += ('\n' if current['content'] else '') + text
                current['page_end'] = page_number
            elif include_unsectioned_content:
                # Last resort: preserve content even when heading typography is
                # inconsistent, rather than returning an almost-empty result.
                current = {
                    'section_name': 'Unsectioned Content',
                    'source_heading': None,
                    'canonical_section': 'unsectioned_content',
                    'page_start': page_number,
                    'page_end': page_number,
                    'content': text,
                }
                sections.append(current)

    for section in sections:
        section['word_count'] = len(re.findall(r'\w+', section['content']))
    return sections

flat_sections = extract_flat_sections(
    context['classified_pages'], is_strict_major_heading
)
# Recover reports that use 11–13 pt bold headings without making that looser
# rule the default for documents that already split correctly.
used_recovery_heading_pass = len(flat_sections) <= 2
if used_recovery_heading_pass:
    flat_sections = extract_flat_sections(
        context['classified_pages'],
        is_recovery_major_heading,
    )
# Only after both heading passes fail do we preserve all content as a named
# fallback instead of returning an almost-empty extraction.
used_all_content_pages_fallback = len(flat_sections) <= 2
if used_all_content_pages_fallback:
    flat_sections = extract_flat_sections(
        context['classified_pages'],
        is_recovery_major_heading,
        include_unsectioned_content=True,
    )
write_json('07_flat_sections.json', flat_sections)
print(f'Recovery heading pass used: {used_recovery_heading_pass}')
print(f'All-content fallback used: {used_all_content_pages_fallback}')
[
    {
        'section': s['section_name'], 'pages': f"{s['page_start']}-{s['page_end']}",
        'words': s['word_count'], 'preview': s['content'][:160],
    }
    for s in flat_sections
]


In [ ]:
# Optional comparison only: run the current production tree extractor and inspect its top-level result.
current_tree_sections = MetadataSectionsStep.extract_project_sections(context['classified_pages'])
write_json('08_current_tree_sections_comparison.json', current_tree_sections)
[
    {
        'heading': s['heading'], 'pages': f"{s['page_start']}-{s['page_end']}",
        'words': s['word_count'], 'children': len(s.get('children', [])),
    }
    for s in current_tree_sections
]


## Batch-run every repository PDF



In [ ]:
PDF_DIRECTORY = REPO_ROOT / 'docs' / 'docs'
BATCH_OUTPUT_DIR = REPO_ROOT / 'notebook_outputs' / 'batch'
PDF_PATHS = sorted(PDF_DIRECTORY.rglob('*.pdf'))

if not PDF_PATHS:
    raise FileNotFoundError(f'No PDF files found under {PDF_DIRECTORY}')

def write_batch_json(output_dir, filename, value):
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / filename
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, default=str),
        encoding='utf-8',
    )

def run_pdf_pipeline(pdf_path):
    relative_pdf = pdf_path.relative_to(PDF_DIRECTORY).with_suffix('')
    output_dir = BATCH_OUTPUT_DIR / relative_pdf
    job = SimpleNamespace(
        file=SimpleNamespace(path=str(pdf_path)),
        file_hash='notebook-batch-debug',
    )
    batch_context = {'start_time': time.perf_counter(), 'draft_metadata': {}}

    for step, context_key, filename in [
        (PDFExtractionStep(), 'raw_pages', '01_raw_pages.json'),
        (LineReconstructionStep(), 'reconstructed_pages', '02_reconstructed_pages.json'),
        (PageClassificationStep(), 'classified_pages', '03_classified_pages.json'),
    ]:
        batch_context = step.run(job, batch_context)
        write_batch_json(output_dir, filename, batch_context[context_key])

    page_summary = [
        {
            'page': page['page_number'],
            'page_type': page.get('page_type'),
            'lines': len(page.get('lines', [])),
            'preview': ' '.join(
                line['text'] for line in page.get('lines', [])[:3]
            )[:160],
        }
        for page in batch_context['classified_pages']
    ]
    write_batch_json(output_dir, '04_page_classification_summary.json', page_summary)

    for step in (MetadataFrontMatterStep(), MetadataPeopleStep()):
        batch_context = step.run(job, batch_context)
    write_batch_json(output_dir, '05_front_matter_and_people.json', {
        'project_fields': batch_context['draft_metadata'].get('project_fields', {}),
        'keywords': batch_context['draft_metadata'].get('keywords', []),
        'people_and_roles': batch_context['draft_metadata'].get('people_and_roles', []),
    })

    pages = batch_context['classified_pages']
    back_matter = MetadataSectionsStep.extract_back_matter(pages)
    write_batch_json(output_dir, '06_back_matter.json', back_matter)

    flat_sections = extract_flat_sections(pages, is_strict_major_heading)
    used_recovery_heading_pass = len(flat_sections) <= 2
    if used_recovery_heading_pass:
        flat_sections = extract_flat_sections(pages, is_recovery_major_heading)
    used_all_content_pages_fallback = len(flat_sections) <= 2
    if used_all_content_pages_fallback:
        flat_sections = extract_flat_sections(
            pages, is_recovery_major_heading, include_unsectioned_content=True
        )
    write_batch_json(output_dir, '07_flat_sections.json', flat_sections)

    tree_sections = MetadataSectionsStep.extract_project_sections(pages)
    write_batch_json(output_dir, '08_current_tree_sections_comparison.json', tree_sections)
    return {
        'pdf': str(pdf_path.relative_to(REPO_ROOT)),
        'output': str(output_dir.relative_to(REPO_ROOT)),
        'pages': len(pages),
        'flat_sections': len(flat_sections),
        'recovery_heading_pass': used_recovery_heading_pass,
        'all_content_fallback': used_all_content_pages_fallback,
        'status': 'ok',
    }

batch_summary = []
for index, pdf_path in enumerate(PDF_PATHS, start=1):
    print(f'[{index}/{len(PDF_PATHS)}] {pdf_path.relative_to(PDF_DIRECTORY)}')
    try:
        batch_summary.append(run_pdf_pipeline(pdf_path))
    except Exception as exc:
        batch_summary.append({
            'pdf': str(pdf_path.relative_to(REPO_ROOT)),
            'status': 'error',
            'error': f'{type(exc).__name__}: {exc}',
        })

write_batch_json(BATCH_OUTPUT_DIR, '_batch_summary.json', batch_summary)
print(f'Completed {len(batch_summary)} PDF(s); results are in {BATCH_OUTPUT_DIR}')
batch_summary
